# 05-01 Pair Explorer

Interactive exploration of the top correlation pairs. Loads backtest metrics from S3 with **DuckDB httpfs**, then plots daily close prices from the minute-summary table for any pair via an interactive dropdown.

In [ ]:
# ============================================================================
# SETUP -- installs, imports, config (env vars / config.json -- never hardcoded)
# ============================================================================

# --- Install packages (no-op if already present) --------------------------
# !pip install -q duckdb ipywidgets matplotlib --upgrade
import json
import os
from io import BytesIO
from pathlib import Path

import boto3
import duckdb
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Configuration --------------------------------------------------------
# Secrets resolve in priority order:
#   1. Environment variables (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY,
#      AWS_REGION, S3_BUCKET, MASSIVE_API_KEY, ...)
#   2. config.json in the current directory (see config.example.json)
#   3. Built-in defaults (non-secret values only)
# On Kaggle: set secrets via notebook settings (Add-ons -> Secrets), which
# are injected as environment variables.
CONFIG_FILE = "config.json"


def get_secret(name, default=""):
    val = os.environ.get(name)
    if val:
        return val
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            if name in data:
                return str(data[name])
        except (OSError, ValueError):
            pass
    return default


class Config:
    def __init__(self):
        self.aws_access_key_id = ""
        self.aws_secret_access_key = ""
        self.aws_region = "us-east-1"
        self.s3_bucket = "market-data-zw"
        self.massive_api_key = ""

        # Paths (S3 keys under the bucket)
        self.types_prefix = "parquet_data/types"
        self.tickers_prefix = "parquet_data/summary/tickers"
        self.ticker_details_prefix = "parquet_data/summary/ticker_yahoo_details"
        self.minute_staging_prefix = "parquet_data/minute_data_staging"
        self.minute_final_prefix = "parquet_data/minute_data_final"
        self.minute_summary_prefix = "parquet_data/summary/minute_summary"
        self.daily_volume_prefix = "parquet_data/summary/daily_volume"
        self.correlation_prefix = "parquet_data/strategies/correlation"
        self.backtest_prefix = "parquet_data/backtest"
        self.backtest_metrics_prefix = "parquet_data/analysis/backtest_metrics"

        # Spark
        self.spark_executor_memory = "24g"
        self.spark_executor_cores = 4
        self.spark_driver_memory = "24g"
        self.spark_tmp = "/tmp/spark"


def load_config():
    cfg = Config()
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            for key, value in data.items():
                if hasattr(cfg, key):
                    setattr(cfg, key, value)
        except (OSError, ValueError) as e:
            print(f"[config] WARNING: could not load {CONFIG_FILE}: {e}")

    env_map = {
        "AWS_ACCESS_KEY_ID": "aws_access_key_id",
        "AWS_SECRET_ACCESS_KEY": "aws_secret_access_key",
        "AWS_REGION": "aws_region",
        "S3_BUCKET": "s3_bucket",
        "MASSIVE_API_KEY": "massive_api_key",
        "SPARK_DRIVER_MEMORY": "spark_driver_memory",
        "SPARK_EXECUTOR_MEMORY": "spark_executor_memory",
        "SPARK_EXECUTOR_CORES": "spark_executor_cores",
    }
    for env_name, attr in env_map.items():
        val = os.environ.get(env_name)
        if val:
            if attr == "spark_executor_cores":
                val = int(val)
            setattr(cfg, attr, val)
    return cfg
# --- S3 helpers -----------------------------------------------------------
def s3_client(cfg):
    from botocore.config import Config as BotocoreConfig
    config = BotocoreConfig(retries={"max_attempts": 5, "mode": "adaptive"},
                            connect_timeout=30, read_timeout=60)
    return boto3.client("s3",
                        aws_access_key_id=cfg.aws_access_key_id,
                        aws_secret_access_key=cfg.aws_secret_access_key,
                        region_name=cfg.aws_region,
                        config=config)


def upload_parquet(df, s3, bucket, key, compression="snappy"):
    buf = BytesIO()
    df.to_parquet(buf, index=False, engine="pyarrow", compression=compression,
                  coerce_timestamps="ms", allow_truncated_timestamps=True)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())


def download_parquet(s3, bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))


def list_s3_keys(s3, bucket, prefix):
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])
    return keys


def tickers_from_prefix(s3, bucket, prefix):
    """Ticker symbols from `<prefix>/<TICKER>.parquet` object keys."""
    return [k.split("/")[-1][:-len(".parquet")] for k in list_s3_keys(s3, bucket, prefix)
            if k.endswith(".parquet")]


def duckdb_s3_connect(cfg):
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute(f"SET s3_access_key_id='{cfg.aws_access_key_id}';")
    con.execute(f"SET s3_secret_access_key='{cfg.aws_secret_access_key}';")
    con.execute(f"SET s3_region='{cfg.aws_region}';")
    return con


def spark_session(cfg):
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .appName("MarketDataPlatform")
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1")
        .config("spark.executor.memory", cfg.spark_executor_memory)
        .config("spark.executor.cores", str(cfg.spark_executor_cores))
        .config("spark.driver.memory", cfg.spark_driver_memory)
        .config("spark.hadoop.fs.s3a.access.key", cfg.aws_access_key_id)
        .config("spark.hadoop.fs.s3a.secret.key", cfg.aws_secret_access_key)
        .config("spark.hadoop.fs.s3a.endpoint", f"s3.{cfg.aws_region}.amazonaws.com")
        .config("spark.local.dir", cfg.spark_tmp)
        .config("spark.hadoop.tmp.dir", cfg.spark_tmp)
        .config("spark.sql.warehouse.dir", f"{cfg.spark_tmp}/warehouse")
        .getOrCreate()
    )
    spark.conf.set("spark.hadoop.fs.s3a.committer.name", "directory")
    spark.conf.set("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    spark.conf.set("spark.hadoop.fs.s3a.committer.staging.conflict-mode", "append")
    spark.conf.set("spark.sql.debug.maxToStringFields", "100")
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    spark.conf.set("spark.sql.ansi.enabled", "false")
    spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
    spark.conf.set("spark.sql.parquet.mergeSchema", "true")
    spark.sparkContext.setLogLevel("ERROR")
    return spark

# --- Instantiate config + clients -----------------------------
cfg = load_config()
s3 = s3_client(cfg)
print("Setup complete")
print(f"Bucket: {cfg.s3_bucket} | Region: {cfg.aws_region}")


In [ ]:
# ============================================================================
# Data source resolution -- prefer the local Kaggle dataset mirror (created
# by 02-02-s3-to-kaggle-dataset: fast, free reads), fall back to S3.
# ============================================================================

import glob as _glob

MIRROR_CANDIDATES = [
    "/kaggle/input/datasets/dsptlp/market-data-s3-dataset/s3_data/parquet_data",
    "/kaggle/input/market-data-s3-dataset/s3_data/parquet_data",
]


def resolve(rel, name="", use_glob=False):
    """Kaggle-mirror path when mounted, else s3a:// URI."""
    short = rel[len("parquet_data/"):] if rel.startswith("parquet_data/") else rel
    for root in MIRROR_CANDIDATES:
        local = os.path.join(root, short, name)
        if use_glob:
            if _glob.glob(local):
                return local
        elif os.path.exists(local):
            return local
    return f"s3://{cfg.s3_bucket}/{rel}/{name}"


In [ ]:
# ============================================================================
# STEP 1 -- Load backtest metrics from S3
# ============================================================================

con = duckdb_s3_connect(cfg)

metrics_path = resolve(cfg.backtest_metrics_prefix, "*/*", use_glob=True)
print("metrics_path:", metrics_path)

metrics = con.execute(f"""
    SELECT *
    FROM read_parquet('{metrics_path}', union_by_name = true)
""").df()

print(f"Loaded {len(metrics):,} pairs from {metrics['run_timestamp'].nunique()} runs")
display(metrics.head(10))

In [ ]:
# ============================================================================
# STEP 2 -- Pick a pair (interactive dropdown)
# ============================================================================

TOP_N = 20
top_pairs = metrics.nlargest(TOP_N, 'robust_score')

print(f"Top {TOP_N} pairs by robust_score:")
display(top_pairs[['leader', 'follower', 'n_trades', 'information_coefficient',
                   'ic_lo_95', 'win_rate', 'mean_profit_pct', 'robust_score']].head(TOP_N))

pairs = (metrics[['leader', 'follower', 'n_trades', 'information_coefficient', 'robust_score']]
         .drop_duplicates()
         .sort_values('robust_score', ascending=False))

options = []
for _, r in pairs.iterrows():
    label = f"{r['leader']} -> {r['follower']}  (n={int(r['n_trades'])}, IC={r['information_coefficient']:.2f})"
    options.append((label, (r['leader'], r['follower'])))

dropdown_options = options[:100]


def plot_pair(label, pair):
    leader, follower = pair
    print(f"\nLoading daily prices for {leader} and {follower}...")
    prices_path = resolve(cfg.minute_summary_prefix, "data.parquet")
    pdf = con.execute(f"""
        SELECT symbol AS ticker, close AS price, trade_date AS date
        FROM read_parquet('{prices_path}')
        WHERE symbol IN ('{leader}', '{follower}') AND rn_desc = 1
        ORDER BY symbol, trade_date
    """).df()
    pdf['date'] = pd.to_datetime(pdf['date'], errors='coerce')
    pdf = pdf.sort_values(['ticker', 'date']).dropna(subset=['date'])
    pivot = pdf.pivot(index='date', columns='ticker', values='price').dropna()
    if leader not in pivot.columns or follower not in pivot.columns:
        print(f"ERROR: Missing price data for {leader} or {follower}")
        return

    norm = pivot / pivot.iloc[0] * 100
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

    ax1.plot(pivot.index, pivot[leader], color='steelblue', linewidth=1.2,
             label=f'{leader} (leader)')
    ax1.plot(pivot.index, pivot[follower], color='coral', linewidth=1.2,
             label=f'{follower} (follower)')
    ax1.set_ylabel('Price ($)')
    ax1.set_title(f'{leader} vs {follower} - Daily Close Prices')
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)

    ax2.plot(norm.index, norm[leader], color='steelblue', linewidth=1.2,
             label=f'{leader} (leader)')
    ax2.plot(norm.index, norm[follower], color='coral', linewidth=1.2,
             label=f'{follower} (follower)')
    ax2.axhline(100, color='black', linewidth=0.5, linestyle='--', alpha=0.5)
    ax2.set_ylabel('Normalized Price (start = 100)')
    ax2.set_xlabel('Date')
    ax2.legend(loc='upper left')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    leader_ret = (pivot[leader].iloc[-1] / pivot[leader].iloc[0] - 1) * 100
    follower_ret = (pivot[follower].iloc[-1] / pivot[follower].iloc[0] - 1) * 100
    print(f"Price change: {leader}: {leader_ret:+.2f}%  |  {follower}: {follower_ret:+.2f}%")

    lr = pivot[leader].pct_change().dropna()
    fr = pivot[follower].pct_change().dropna()
    aligned = pd.concat([lr, fr], axis=1, keys=['leader', 'follower']).dropna()
    print(f"Daily return correlation: {aligned['leader'].corr(aligned['follower']):.4f}")


w = widgets.Dropdown(options=dropdown_options, description='Pair:',
                     layout=widgets.Layout(width='600px'))
display(w)
w.observe(lambda change: plot_pair(change['new'][0], change['new'][1]), names='value')
plot_pair(dropdown_options[0][0], dropdown_options[0][1])

In [ ]:
# ============================================================================
# STEP 3 -- Static gallery: top 10 pairs
# ============================================================================

TOP_GALLERY = 10
print(f"Plotting top {TOP_GALLERY} pairs...")
for i, (label, pair) in enumerate(dropdown_options[:TOP_GALLERY]):
    print(f"\n--- {i + 1}. {label} ---")
    plot_pair(label, pair)